# **國立中興大學 研究方法 (Research Methodology) - 期末專案**
## **Part 2: YOLOv7 台灣市區交通違規即時偵測與消融研究**

---
### 📌 **專案基本資訊**
* **專題題目**：基於 YOLOv7 深度學習之台灣多類別交通違規停車即時偵測與動態影片分析
* **繳交期限 (Deadline)**：**2026/12/17**
* **成果繳交格式**：`{StudentID1}_{StudentID2}_{StudentID3}.zip`（含報告 Slides / 文件）
* **4 大違規標註類別 (Target Classes)**：
  0. `red_line_parking` (紅線違規停車)
  1. `double_parking` (並排違規停車)
  2. `yellow_box_parking` (黃網線臨時暫停)
  3. `sidewalk_parking` (人行道/騎樓違停)

* **模型倉庫**：`contemplator/nchu-research-method-traffic-violation-model`
* **資料/實驗倉庫**：`contemplator/nchu-research-method-traffic-violation-dataset`

# **Step 1: 環境檢查與 GPU 配置 (Environment Checking)**

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Current Device: {torch.cuda.get_device_name(0)}")

# **Step 2: 掛載 Google 雲端硬碟 (Mount Google Drive)**

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# **Step 3: 下載專案倉庫與安裝 YOLOv7 依賴 (Setup YOLOv7)**
> 若為 Private 倉庫，可填寫您的 GitHub Personal Access Token (PAT) 以便免密碼 Clone。

In [ ]:
%cd /content
!mkdir -p workspace
%cd /content/workspace

# 選擇一：若為 Public 或已設定 Token 存取 Private
# GITHUB_TOKEN = "your_pat_here"  # 若為私有庫可取消註解填入
!git clone https://github.com/contemplator/nchu-research-method-traffic-violation-model.git yolov7
!git clone https://github.com/contemplator/nchu-research-method-traffic-violation-dataset.git dataset_repo

# 安裝依賴套件
%cd /content/workspace/yolov7
!pip install -r requirements.txt
!pip install albumentations

## **下載官方預訓練權重 (Pretrained Weights)**

In [ ]:
%cd /content/workspace/yolov7
!wget -P /content/workspace/yolov7 https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7-tiny.pt
!wget -P /content/workspace/yolov7 https://github.com/WongKinYiu/yolov7/releases/download/v0.1/yolov7.pt

# **Step 4: 準備資料集 (Dataset Preparation)**
將本地打包切分好的 `dataset.zip` 自 Google Drive 解壓縮至工作區：

In [ ]:
import os
# 請確認 Drive 上的 zip 路徑是否正確 (例如 Drive 中的 NCHU_YOLO_Project/dataset.zip)
DRIVE_DATASET_ZIP = "/content/drive/MyDrive/NCHU_YOLO_Project/dataset.zip"
TARGET_DATASET_DIR = "/content/workspace/dataset_repo/data/dataset"

if os.path.exists(DRIVE_DATASET_ZIP):
    !unzip -q {DRIVE_DATASET_ZIP} -d {TARGET_DATASET_DIR}
    print("✅ 資料集自 Google Drive 解壓縮完成！")
else:
    print(f"⚠️ 尚未在 {DRIVE_DATASET_ZIP} 找到壓縮檔，若資料已存在於 dataset_repo 中則可直接使用。")

# 檢查 data.yaml
!cat /content/workspace/dataset_repo/configs/traffic_violation.yaml

# **Step 5: 4 組消融實驗訓練 (Ablation Experiments)**

| 組別 | 模型架構 | Epochs | Batch | 目的與探討目標 |
| :---: | :--- | :---: | :---: | :--- |
| **Exp 1** | YOLOv7-tiny | 50 | 16 | 輕量輕量模型基準線 (Baseline) |
| **Exp 2** | YOLOv7 Base | 50 | 8 | 骨幹模型尺度對複雜遮擋與小目標效益 |
| **Exp 3** | YOLOv7-tiny | 100 | 16 | 延長訓練週期之充分收斂性與過擬合檢視 |
| **Exp 4** | YOLOv7-tiny (Tuned) | 100 | 16 | 超參數調校 (Mosaic 增強 + 學習率微調) |

### **🔥 Exp 1: YOLOv7-tiny 輕量基準線 (50 Epochs)**

In [ ]:
%cd /content/workspace/yolov7
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 16 \
  --epochs 50 \
  --data /content/workspace/dataset_repo/configs/traffic_violation.yaml \
  --img 640 640 \
  --cfg /content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml \
  --weights yolov7-tiny.pt \
  --name exp1_yolov7_tiny_baseline \
  --hyp data/hyp.scratch.tiny.yaml

### **🔥 Exp 2: YOLOv7 Base 尺度對比 (50 Epochs)**

In [ ]:
%cd /content/workspace/yolov7
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 8 \
  --epochs 50 \
  --data /content/workspace/dataset_repo/configs/traffic_violation.yaml \
  --img 640 640 \
  --cfg cfg/training/yolov7.yaml \
  --weights yolov7.pt \
  --name exp2_yolov7_base_scale \
  --hyp data/hyp.scratch.p5.yaml

### **🔥 Exp 3: 充分收斂週期對比 (100 Epochs)**

In [ ]:
%cd /content/workspace/yolov7
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 16 \
  --epochs 100 \
  --data /content/workspace/dataset_repo/configs/traffic_violation.yaml \
  --img 640 640 \
  --cfg /content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml \
  --weights yolov7-tiny.pt \
  --name exp3_yolov7_tiny_100ep \
  --hyp data/hyp.scratch.tiny.yaml

### **🔥 Exp 4: 超參數與資料增強微調 (Custom Hyp + 100 Epochs)**

In [ ]:
%cd /content/workspace/yolov7
!python train.py \
  --workers 4 \
  --device 0 \
  --batch-size 16 \
  --epochs 100 \
  --data /content/workspace/dataset_repo/configs/traffic_violation.yaml \
  --img 640 640 \
  --cfg /content/workspace/dataset_repo/configs/yolov7-tiny-custom.yaml \
  --weights yolov7-tiny.pt \
  --name exp4_yolov7_tiny_tuned \
  --hyp /content/workspace/dataset_repo/configs/hyp.scratch.custom.yaml

# **Step 6: 測試集獨立指標評估 (Test Evaluation)**

In [ ]:
%cd /content/workspace/yolov7
# 評估 Exp 4 於測試集 (test split) 之表現
!python test.py \
  --data /content/workspace/dataset_repo/configs/traffic_violation.yaml \
  --img 640 \
  --batch 16 \
  --conf 0.001 \
  --iou 0.65 \
  --device 0 \
  --weights runs/train/exp4_yolov7_tiny_tuned/weights/best.pt \
  --name exp4_test_evaluation

# **Step 7: 影像與動態行車記錄器影片推論 (Inference)**

In [ ]:
%cd /content/workspace/yolov7

# 1. 測試集圖片推論
!python detect.py \
  --weights runs/train/exp4_yolov7_tiny_tuned/weights/best.pt \
  --conf 0.25 \
  --img-size 640 \
  --source /content/workspace/dataset_repo/data/dataset/test/images/ \
  --name detect_test_images

# 2. 行車記錄器影片推論 (若有提供 sample_videos/dashcam.mp4)
VIDEO_PATH = "/content/workspace/dataset_repo/sample_videos/dashcam_nchu.mp4"
if os.path.exists(VIDEO_PATH):
    !python detect.py \
      --weights runs/train/exp4_yolov7_tiny_tuned/weights/best.pt \
      --conf 0.35 \
      --img-size 640 \
      --source {VIDEO_PATH} \
      --name detect_video_dashcam

# **Step 8: 成果圖表視覺化 (Visualizations)**

In [ ]:
import glob
import matplotlib.pyplot as plt
import cv2

# 顯示 Exp 4 訓練曲線與混淆矩陣
exp4_results = "runs/train/exp4_yolov7_tiny_tuned/results.png"
exp4_cm = "runs/train/exp4_yolov7_tiny_tuned/confusion_matrix.png"

if os.path.exists(exp4_results):
    plt.figure(figsize=(14, 8), dpi=150)
    img = cv2.imread(exp4_results)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Exp 4 Training Results & Metrics")
    plt.axis("off")
    plt.show()

if os.path.exists(exp4_cm):
    plt.figure(figsize=(10, 8), dpi=150)
    img = cv2.imread(exp4_cm)
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title("Exp 4 Confusion Matrix")
    plt.axis("off")
    plt.show()

In [ ]:
# 繪製推論成果抽樣網格 (2x3)
det_images = glob.glob("runs/detect/detect_test_images/*.jpg")[:6]
if det_images:
    fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=150)
    axes = axes.flatten()
    for i, img_path in enumerate(det_images):
        img = cv2.imread(img_path)
        axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        axes[i].set_title(os.path.basename(img_path), fontsize=10)
        axes[i].axis("off")
    plt.tight_layout()
    plt.show()

# **Step 9: 儲存權重與期末成果打包 (Save to Google Drive)**
自動將各組權重、指標圖表與推論成果打包壓縮至 Google Drive 備份，供期末報告投影片使用。

In [ ]:
!mkdir -p /content/drive/MyDrive/NCHU_YOLO_Project/results
!cp runs/train/exp4_yolov7_tiny_tuned/weights/best.pt /content/drive/MyDrive/NCHU_YOLO_Project/results/best_model_exp4.pt

# 打包所有實驗圖表與推論成果
!zip -r /content/drive/MyDrive/NCHU_YOLO_Project/results_summary.zip \
  runs/train/exp*/results.png \
  runs/train/exp*/confusion_matrix.png \
  runs/train/exp*/weights/best.pt \
  runs/detect/

print("🎉 實驗指標與權重已成功備份至 Google Drive (NCHU_YOLO_Project/)！")